In [7]:
from pathlib import Path
import json
import pandas as pd

In [9]:
project_dir = Path.cwd()
data_dir = project_dir / "data"

print("Project folder:", project_dir)
print("Data folder exists:", data_dir.exists())

Project folder: C:\Users\sonal\OneDrive\Desktop\case_study_jp
Data folder exists: True


In [11]:
for file in sorted(data_dir.rglob("*")):
    if file.is_file():
        print(file.relative_to(data_dir))

atlas\atlas_usage_20260817.xlsx
atlas\atlas_usage_20260824.xlsx
atlas\atlas_usage_20260831.xlsx
atlas\atlas_usage_20260907.xlsx
atlas\atlas_usage_20260913.xlsx
atlas\atlas_usage_20260914.xlsx
atlas\atlas_usage_20260921.xlsx
atlas\atlas_usage_20261005.xlsx
hr_roster.csv
quill_events.csv
reporting_config.json


In [13]:
with open(data_dir / "reporting_config.json", encoding="utf-8") as file:
    config = json.load(file)

print(json.dumps(config, indent=4))    # json string

{
    "reporting_window": {
        "first_week_start": "2026-08-09",
        "last_week_start": "2026-09-27"
    },
    "week_definition": "Weeks run Sunday 00:00 UTC to Saturday 23:59:59 UTC. week_start is the Sunday date.",
    "scope": {
        "division": "ADV",
        "division_name": "Advisory",
        "hr_rule": "Use each person's most recent HR record (latest Effective_Date) for every week."
    },
    "tools": [
        {
            "tool_id": "quill",
            "source": "quill_events.csv",
            "wau_metric": "count of event_type == 'generation'",
            "wau_threshold": 1,
            "wpu_threshold": 12
        },
        {
            "tool_id": "atlas",
            "source": "atlas/atlas_usage_<YYYYMMDD extract date>.xlsx",
            "wau_metric": "Active Days",
            "wau_threshold": 1,
            "wpu_metric": "Queries Run",
            "wpu_threshold": 25
        }
    ]
}


In [15]:
print("First week:", config["reporting_window"]["first_week_start"])
print("Last week:", config["reporting_window"]["last_week_start"])
print("Required division:", config["scope"]["division"])

First week: 2026-08-09
Last week: 2026-09-27
Required division: ADV


**loading Data - Hr Roster**

**Data Inspection**

In [23]:
hr = pd.read_csv(data_dir / "hr_roster.csv")    #loading Hr Roter data set

hr.head()

,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
0,E8456729,ADV,People Advisory,AMER,L5,2023-11-13
1,E3961175,ADV,Strategy,APAC,L1,2024-09-19
2,E2473900,AUD,Audit Assurance,AMER,L5,2023-02-22
3,E9720776,TAX,Corporate Tax,AMER,L4,2026-04-11
4,E5492817,SHS,IT Services,AMER,L2,2023-01-16


In [ ]:
print("Rows and columns:", hr.shape)
print("Unique employees:", hr["Employee_ID"].nunique())

hr.info()

In [30]:
print("Rows and columns:", hr.shape)                         # how many rows & columns
print("Unique employees:", hr["Employee_ID"].nunique())

hr.info()

Rows and columns: (2445, 6)
Unique employees: 2400
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2445 entries, 0 to 2444
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     2445 non-null   object
 1   Division        2445 non-null   object
 2   Department      2445 non-null   object
 3   Region          2445 non-null   object
 4   Job_Level       2445 non-null   object
 5   Effective_Date  2445 non-null   object
dtypes: object(6)
memory usage: 114.7+ KB


In [32]:
print("Missing values by column:")
print(hr.isna().sum())

print("\nExact duplicate rows:", hr.duplicated().sum())              # every column matches another row

Missing values by column:
Employee_ID       0
Division          0
Department        0
Region            0
Job_Level         0
Effective_Date    0
dtype: int64

Exact duplicate rows: 0


In [38]:
multiple_records = hr[hr.duplicated(subset="Employee_ID", keep=False)]   # keep=False = marks all occurrences of a repeated employee ID

multiple_records.sort_values(["Employee_ID", "Effective_Date"]).head(10)

,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
1730,E0162274,SHS,Finance Operations,EMEA,L3,2026-01-08
2053,E0162274,ADV,Risk,EMEA,L3,2026-07-12
944,E0171370,ADV,Technology Consulting,AMER,L1,2023-08-14
369,E0171370,ADV,People Advisory,AMER,L1,2026-09-01
1272,E0588098,ADV,Strategy,AMER,L1,2025-09-14
1208,E0588098,ADV,Deals,AMER,L1,2026-09-01
33,E0668929,ADV,Deals,AMER,L4,2025-12-29
924,E0668929,SHS,IT Services,AMER,L4,2026-08-14
492,E0907126,AUD,Audit Assurance,AMER,L4,2024-08-26
1451,E0907126,ADV,Deals,AMER,L4,2026-08-19


## HR data preparation
Standardise employee IDs, select each employee's latest HR record,
and retain employees in the Advisory division.

In [41]:
# Keep the original HR data unchanged for comparison.

hr_clean = hr.copy()

In [43]:
# Remove surrounding spaces and convert IDs to uppercase.

hr_clean["Employee_ID"] = hr_clean["Employee_ID"].str.strip().str.upper()


hr_clean.head()

,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
0,E8456729,ADV,People Advisory,AMER,L5,2023-11-13
1,E3961175,ADV,Strategy,APAC,L1,2024-09-19
2,E2473900,AUD,Audit Assurance,AMER,L5,2023-02-22
3,E9720776,TAX,Corporate Tax,AMER,L4,2026-04-11
4,E5492817,SHS,IT Services,AMER,L2,2023-01-16


In [45]:
# Convert text dates into actual dates so we can order records correctly.
# The source format is year-month-day, for example 2026-08-14.

hr_clean["Effective_Date"] = pd.to_datetime(hr_clean["Effective_Date"],format="%Y-%m-%d")

# Check that Effective_Date now has a datetime type.
hr_clean.dtypes

Employee_ID               object
Division                  object
Department                object
Region                    object
Job_Level                 object
Effective_Date    datetime64[ns]
dtype: object

In [47]:
# Group each employee's records together and order them by date.

hr_sorted = hr_clean.sort_values(by=["Employee_ID", "Effective_Date"])

hr_sorted.head(10)

,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
1831,E0001338,AUD,Audit Assurance,EMEA,L1,2026-01-07
2150,E0005667,ADV,Technology Consulting,APAC,L3,2025-05-07
2334,E0006083,ADV,People Advisory,APAC,L3,2024-03-20
2064,E0006249,ADV,People Advisory,EMEA,L2,2024-07-02
2058,E0014152,SHS,Finance Operations,AMER,L3,2026-04-08
1147,E0017798,ADV,Risk,APAC,L1,2024-03-12
421,E0024994,ADV,Deals,AMER,L1,2024-01-07
1984,E0028755,ADV,Strategy,AMER,L4,2024-04-10
185,E0034251,ADV,Strategy,AMER,L3,2023-09-07
1586,E0039658,ADV,People Advisory,APAC,L1,2023-04-25


In [49]:
# Because dates are sorted oldest to newest, keep="last"
# retains the most recent record for each employee.

hr_latest = hr_sorted.drop_duplicates(subset="Employee_ID",keep="last")

print("Rows before selecting latest records:", len(hr_sorted))
print("Rows after selecting latest records:", len(hr_latest))

Rows before selecting latest records: 2445
Rows after selecting latest records: 2400


Here, drop_duplicates() checks only Employee_ID because we specified subset. We are deliberately selecting one record per employee—not claiming their historical records were mistakes.

In [58]:
# Read the required division from the configuration.
required_division = config["scope"]["division"]

# Filter only AFTER selecting each employee's latest record.
hr_advisory = hr_latest[hr_latest["Division"] == required_division].copy()

print("Eligible Advisory employees:", len(hr_advisory))

hr_advisory.head()

Eligible Advisory employees: 1507


,Employee_ID,Division,Department,Region,Job_Level,Effective_Date
2150,E0005667,ADV,Technology Consulting,APAC,L3,2025-05-07
2334,E0006083,ADV,People Advisory,APAC,L3,2024-03-20
2064,E0006249,ADV,People Advisory,EMEA,L2,2024-07-02
1147,E0017798,ADV,Risk,APAC,L1,2024-03-12
421,E0024994,ADV,Deals,AMER,L1,2024-01-07


question - Why this order? If someone moved from ADV to TAX, their latest TAX record must determine eligibility. Filtering to ADV first would wrongly retain their old record.

In [61]:
# Each employee should appear only once.
print("Repeated employee IDs:", hr_advisory["Employee_ID"].duplicated().sum())

# Every remaining record should belong to ADV.
print("Remaining divisions:", hr_advisory["Division"].unique())

Repeated employee IDs: 0
Remaining divisions: ['ADV']


In [63]:
hr_advisory["Division"].value_counts()

Division
ADV    1507
Name: count, dtype: int64

## Quill usage data
Explore the event log and prepare it for weekly user calculations.

In [66]:
# Load the Quill event log.
quill = pd.read_csv(data_dir / "quill_events.csv")

quill.head()

,user_id,event_time_utc,event_type,client
0,E6616097,2026-08-02T00:02:08Z,generation,mobile
1,E3050267,2026-08-02T00:12:28Z,generation,desktop
2,E3050267,2026-08-02T00:12:28Z,generation,desktop
3,E7932530,2026-08-02T00:18:01Z,generation,web
4,E6113456,2026-08-02T00:19:41Z,generation,desktop


In [68]:
# Check the number of rows, columns and their data types.
print("Rows and columns:", quill.shape)

quill.info()

Rows and columns: (94558, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94558 entries, 0 to 94557
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   user_id         94558 non-null  object
 1   event_time_utc  94558 non-null  object
 2   event_type      94558 non-null  object
 3   client          94558 non-null  object
dtypes: object(4)
memory usage: 2.9+ MB


In [74]:
# Count how many records belong to each type of action.

quill["event_type"].value_counts()

event_type
generation    72857
open          14405
rating         7296
Name: count, dtype: int64

In [72]:
# Check for missing values in each column.

quill.isna().sum()

user_id           0
event_time_utc    0
event_type        0
client            0
dtype: int64

generation = counted towards Quill’s active and power-user thresholds

open & no-generation = not an active user


In [77]:
# Preserve the original data and work on a copy.

quill_clean = quill.copy()

In [79]:
quill_clean.head()

,user_id,event_time_utc,event_type,client
0,E6616097,2026-08-02T00:02:08Z,generation,mobile
1,E3050267,2026-08-02T00:12:28Z,generation,desktop
2,E3050267,2026-08-02T00:12:28Z,generation,desktop
3,E7932530,2026-08-02T00:18:01Z,generation,web
4,E6113456,2026-08-02T00:19:41Z,generation,desktop


In [81]:
# Standardise IDs so they match the format used in HR.

quill_clean["user_id"] = quill_clean["user_id"].str.strip().str.upper()

In [83]:
quill_clean["user_id"].head(2)

0    E6616097
1    E3050267
Name: user_id, dtype: object

In [85]:
# Count records where the ID formatting changed.

changed_ids = (quill["user_id"] != quill_clean["user_id"]).sum()

print("Rows with ID formatting changes:", changed_ids)

Rows with ID formatting changes: 8455


In [95]:
# Count exact duplicate rows in the original file.
print("Raw duplicate rows:", quill.duplicated().sum())

Raw duplicate rows: 1397


In [97]:
# Check again after standardising IDs.
print("Duplicate rows after ID cleaning:",quill_clean.duplicated().sum())

Duplicate rows after ID cleaning: 1397


In [103]:
# Display all occurrences of duplicate rows so we can inspect them.

duplicate_events = quill_clean[quill_clean.duplicated(keep=False)]

duplicate_events.sort_values(["user_id", "event_time_utc"]).head(10)

,user_id,event_time_utc,event_type,client
88759,E0014152,2026-09-30T21:26:53Z,generation,web
88760,E0014152,2026-09-30T21:26:53Z,generation,web
734,E0024994,2026-08-04T07:48:42Z,generation,mobile
735,E0024994,2026-08-04T07:48:42Z,generation,mobile
14228,E0024994,2026-08-17T02:49:36Z,generation,mobile
14229,E0024994,2026-08-17T02:49:36Z,generation,mobile
64821,E0024994,2026-09-17T11:28:26Z,generation,desktop
64822,E0024994,2026-09-17T11:28:26Z,generation,desktop
69696,E0049329,2026-09-21T00:07:24Z,generation,web
69697,E0049329,2026-09-21T00:07:24Z,generation,web
